In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Simplified: a "Teacher" dual-encoder (stand-in for CLIP) and a small
# uninitialized "Student" that we compress it into.

class DualEncoder(nn.Module):
    def __init__(self, img_dim=512, txt_dim=512, embed_dim=128, hidden=256):
        super().__init__()
        self.image_encoder = nn.Sequential(nn.Linear(img_dim, hidden), nn.ReLU(), nn.Linear(hidden, embed_dim))
        self.text_encoder = nn.Sequential(nn.Linear(txt_dim, hidden), nn.ReLU(), nn.Linear(hidden, embed_dim))

    def forward(self, images, texts):
        img_emb = F.normalize(self.image_encoder(images), dim=-1)
        txt_emb = F.normalize(self.text_encoder(texts), dim=-1)
        return img_emb, txt_emb

def distillation_loss(student_img, student_txt, teacher_img, teacher_txt, temperature=2.0, alpha=0.5):
    # 1) direct cosine-distance loss between student and teacher embeddings
    cosine_loss = (1 - F.cosine_similarity(student_img, teacher_img)).mean() + \
                  (1 - F.cosine_similarity(student_txt, teacher_txt)).mean()

    # 2) KL-divergence between teacher and student image-text similarity logits
    teacher_logits = (teacher_img @ teacher_txt.T) / temperature
    student_logits = (student_img @ student_txt.T) / temperature

    teacher_probs = F.softmax(teacher_logits, dim=-1)
    student_log_probs = F.log_softmax(student_logits, dim=-1)
    kl_loss = F.kl_div(student_log_probs, teacher_probs, reduction="batchmean") * (temperature ** 2)

    return alpha * cosine_loss + (1 - alpha) * kl_loss

def train_distillation(steps=5, batch_size=16, img_dim=512, txt_dim=512):
    teacher = DualEncoder(img_dim, txt_dim, embed_dim=128, hidden=256)
    student = DualEncoder(img_dim, txt_dim, embed_dim=128, hidden=64)  # lightweight student

    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad = False   # frozen "Teacher"

    optimizer = torch.optim.Adam(student.parameters(), lr=1e-3)

    for step in range(steps):
        images = torch.randn(batch_size, img_dim)
        texts = torch.randn(batch_size, txt_dim)

        with torch.no_grad():
            t_img, t_txt = teacher(images, texts)
        s_img, s_txt = student(images, texts)

        loss = distillation_loss(s_img, s_txt, t_img, t_txt)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print(f"step {step}: distillation loss={loss.item():.4f}")

    return student

if __name__ == "__main__":
    torch.manual_seed(0)
    train_distillation()

---
## Task 12: Knowledge Distillation of Dual-Encoder Vision-Language Models